# 1. Previsión de ventas

**Pregunta:** ¿cuánto venderá la empresa los próximos 12 meses?

**Datos:** ventas netas mensuales (`v_ventas_mensuales`), 32 meses.

**Método:**
1. Se reservan los **6 últimos meses** (marzo–agosto 2026) para medir el error. Es lo que pide la Tabla 5 de la memoria: *error de previsión medido sobre un periodo reservado*.
2. Se comparan 4 modelos, de más simple a más complejo. Un modelo solo vale si mejora a los simples.
3. Como 6 meses es poca prueba, se repite la evaluación desde varios puntos de partida (**origen móvil**).
4. Con el mejor modelo, reentrenado con los 32 meses, se prevén los 12 meses siguientes con un intervalo de confianza.

**Métrica:** MAPE (error porcentual absoluto medio). Un MAPE del 10 % significa que, de media, la previsión de un mes se desvía un 10 % de lo real.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

from statsmodels.tsa.holtwinters import ExponentialSmoothing

## 1. La serie y la partición entrenamiento / prueba

In [ ]:
vm = q('SELECT anio_mes, ventas_netas_pen FROM v_ventas_mensuales ORDER BY anio_mes')
serie = pd.Series(vm.ventas_netas_pen.values, index=pd.PeriodIndex(vm.anio_mes, freq='M'))

RESERVA = 6
entren, prueba = serie[:-RESERVA], serie[-RESERVA:]
print(f'Entrenamiento: {entren.index[0]} a {entren.index[-1]} ({len(entren)} meses)')
print(f'Prueba:        {prueba.index[0]} a {prueba.index[-1]} ({len(prueba)} meses)')

## 2. Los modelos candidatos

- **Ingenuo estacional:** cada mes vende lo mismo que el mismo mes del año anterior. Es la referencia mínima.
- **Ingenuo estacional con crecimiento:** lo mismo, multiplicado por lo que creció el último año.
- **Holt-Winters:** suavizado exponencial con *nivel* (cuánto se vende ahora), *tendencia* (amortiguada, para no extrapolar el crecimiento sin freno) y *estacionalidad* de 12 meses. Se prueba con estacionalidad **aditiva** (el pico de diciembre suma una cantidad fija) y **multiplicativa** (suma un porcentaje, y crece si crece la empresa).

No se usa ARIMA ni redes neuronales: con 32 meses no hay datos para estimar tantos parámetros, y un modelo complejo con poca historia suele prever peor que uno simple.

In [ ]:
def holt_winters(y, estacion='mul'):
    return ExponentialSmoothing(y, trend='add', damped_trend=True, seasonal=estacion, seasonal_periods=12).fit()


def ingenuo_estacional(y, h):
    meses = pd.period_range(y.index[-1] + 1, periods=h, freq='M')
    return np.array([y[m - 12] for m in meses])


def ingenuo_crecimiento(y, h):
    return ingenuo_estacional(y, h) * y[-12:].sum() / y[-24:-12].sum()


candidatos = {
    'Ingenuo estacional': lambda y, h: ingenuo_estacional(y, h),
    'Ingenuo estacional con crecimiento': lambda y, h: ingenuo_crecimiento(y, h),
    'Holt-Winters aditivo': lambda y, h: holt_winters(y, 'add').forecast(h).values,
    'Holt-Winters multiplicativo': lambda y, h: holt_winters(y, 'mul').forecast(h).values,
}

## 3. Evaluación

- **Periodo reservado:** cada modelo se entrena hasta febrero de 2026 y prevé marzo–agosto.
- **Origen móvil:** se entrena hasta el mes 24, se prevén los 3 siguientes; luego hasta el 25, y así hasta 6 veces. Promedia 18 errores en lugar de 6, así que la elección no depende de un solo periodo. Holt-Winters necesita al menos dos años completos para arrancar, por eso empieza en el mes 24.

In [ ]:
filas = []
for nombre, prever in candidatos.items():
    p = prever(entren, RESERVA)
    errores = []
    for origen in range(24, len(serie) - 2):
        real = serie[origen:origen + 3].values
        errores += list(np.abs(real - prever(serie[:origen], 3)) / real)
    filas.append({'modelo': nombre,
                  'mape_reserva_pct': round(mape(prueba, p), 1),
                  'error_total_reserva_pct': round((p.sum() / prueba.sum() - 1) * 100, 1),
                  'mape_origen_movil_pct': round(np.mean(errores) * 100, 1)})

comparativa = pd.DataFrame(filas).sort_values('mape_origen_movil_pct')
guardar(comparativa, 'prevision_comparativa_modelos')
comparativa

**Cómo leerlo:** el modelo elegido es el de menor MAPE en origen móvil. `error_total_reserva_pct` dice si, sumando los 6 meses, el modelo se queda corto (negativo) o se pasa (positivo).

In [ ]:
mejor = comparativa.iloc[0].modelo
print('Modelo elegido:', mejor)
prev_reserva = candidatos[mejor](entren, RESERVA)
pd.DataFrame({'real': prueba.values.round(0), 'prevision': prev_reserva.round(0),
              'error_pct': ((prev_reserva / prueba.values - 1) * 100).round(1)}, index=prueba.index.astype(str))

## 4. Previsión de los próximos 12 meses

Se reentrena el modelo elegido con los 32 meses. El **intervalo del 80 %** se obtiene simulando 1.000 futuros posibles con los errores del propio modelo: en 8 de cada 10 simulaciones la venta del mes cae dentro de la banda.

In [ ]:
H = 12
modelo = holt_winters(serie, 'mul' if 'multiplicativo' in mejor else 'add')
meses_futuros = pd.period_range(serie.index[-1] + 1, periods=H, freq='M')
prevision_12 = modelo.forecast(H).values
simulaciones = modelo.simulate(H, repetitions=1000, error='mul', anchor='end', random_state=SEMILLA)
lim_inf = np.percentile(simulaciones, 10, axis=1)
lim_sup = np.percentile(simulaciones, 90, axis=1)

ultimos_12 = serie[-12:].sum()
print(f'Ventas de los últimos 12 meses:  S/ {ultimos_12:,.0f}')
print(f'Previsión de los próximos 12:    S/ {prevision_12.sum():,.0f}  ({(prevision_12.sum() / ultimos_12 - 1) * 100:+.1f} %)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(serie.index.to_timestamp(), serie.values / 1000, color=AZUL, lw=2, label='Ventas reales')
ax.plot(prueba.index.to_timestamp(), prev_reserva / 1000, color=NARANJA, lw=1.6, ls='--',
        label=f'Previsión en el periodo reservado (MAPE {mape(prueba, prev_reserva):.0f} %)')
ax.plot(meses_futuros.to_timestamp(), prevision_12 / 1000, color=AZUL, lw=2, ls=':', label='Previsión 12 meses')
ax.fill_between(meses_futuros.to_timestamp(), lim_inf / 1000, lim_sup / 1000, color=AZUL, alpha=.15, label='Intervalo del 80 %')
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
estilo(ax, 'Ventas netas mensuales y previsión a 12 meses', 'miles de S/ (sin IGV)')
ax.legend(frameon=False, fontsize=8, loc='upper left')
guardar_figura(fig, '1_prevision_ventas')
plt.show()

## 5. Guardar el resultado para el cuadro de mando

Una sola tabla con la serie real, la validación y la previsión. En Power BI se pinta con `anio_mes` en el eje y las tres columnas de importe como series.

In [ ]:
prevision = pd.concat([
    pd.DataFrame({'anio_mes': serie.index.astype(str), 'tipo': 'REAL', 'ventas_reales_pen': serie.values}),
    pd.DataFrame({'anio_mes': prueba.index.astype(str), 'tipo': 'VALIDACION', 'ventas_reales_pen': prueba.values,
                  'prevision_pen': prev_reserva}),
    pd.DataFrame({'anio_mes': meses_futuros.astype(str), 'tipo': 'PREVISION', 'prevision_pen': prevision_12,
                  'limite_inferior_80_pen': lim_inf, 'limite_superior_80_pen': lim_sup}),
], ignore_index=True).round(2)
prevision['modelo'] = mejor
guardar(prevision, 'prevision_ventas')
prevision.tail(12)

## Conclusiones (para la memoria)

- Holt-Winters con estacionalidad multiplicativa es el que menos se equivoca y mejora a las referencias ingenuas, así que el modelo aporta algo frente a "lo mismo que el año pasado".
- Un error de en torno al 10–13 % mensual es razonable para una pyme con pedidos grandes y puntuales: un solo proyecto mueve el mes. Sumando el trimestre o el año, el error es menor.
- En el periodo reservado el modelo se quedó corto (julio y agosto de 2026 fueron excepcionalmente altos). Conviene reentrenarlo cada mes.
- **Limitación:** 32 meses son menos de tres ciclos anuales. La estacionalidad estimada es aproximada y el intervalo de confianza es ancho a 12 meses.